# Nurtec Copay Fraud Signals — 01 · Train models

**Purpose.** Fit and persist every ML model the scoring notebook needs:

1. **IsolationForest** composite anomaly model (+ scaler & score bounds)
2. **RandomForest lookalike classifier** — learns the *profile* of a rule-flagged pharmacy
3. **Per-signal propensity models** — one-vs-rest per signal with enough positives

The deterministic **rule engine** (signals 1–5, 7) runs here too — not because we train it,
but because its flags are the *labels* the supervised models learn from. Everything is fit
on the full history and written to `models/` for the scoring notebook to load.

> All rule logic and feature code live in `nurtec_pipeline.py`, shared with notebook 02,
> so the signal definitions can never drift between training and scoring.

## 1. Setup

In [24]:
import importlib
import nurtec_pipeline
importlib.reload(nurtec_pipeline)
from nurtec_pipeline import *   # re-pull names into your namespace
print(COPAY_PATH)

COPAY_EUCRISA_CLEAN.csv


In [25]:
from nurtec_pipeline import *   # rule engine, features, fit_*/apply_* helpers, CONFIG
import pandas as pd, numpy as np
from datetime import datetime

## 2. Load data

In [26]:
print(COPAY_PATH)

COPAY_EUCRISA_CLEAN.csv


In [15]:
df = pd.read_csv(COPAY_PATH, low_memory=False)
df.head()

,PFZ_CUST_ID,PFZ_PROD_ID,PFZ_PROMO_PROD_ID,PRESC_TYPE,PRESC_ID,PRESC_FNAME,PRESC_LNAME,RX_NBR,CLAIM_NBR,ADJ_CLAIM_NBR,...,CASE QTY,DOSE QTY,WAC_SHIP,ORIGINAL INDICATOR,PROD_DTL_KEY,POC PARENT,_ship_name,_ship_zip,_ship_month,_ship_year
0,4508538,1445994,-1.0,NPI,1801099890,AMY,BOYD,717784,112846095,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1729392,1445994,-1.0,DEA,MB0980664,SHERI,BEAL,1381738,76886603,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1594039,1445994,-1.0,DEA,BD1332016,BARRY,DUBNER,935785,99110503,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1312858,1445994,-1.0,DEA,BW8855910,DONNA,WARD,6610413,81638758,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,147949205,1445994,-1.0,NPI,1083997142,SMITA,AGGARWAL,1629280,77122349,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns

Index(['PFZ_CUST_ID', 'PFZ_PROD_ID', 'PFZ_PROMO_PROD_ID', 'PRESC_TYPE',
       'PRESC_ID', 'PRESC_FNAME', 'PRESC_LNAME', 'RX_NBR', 'CLAIM_NBR',
       'ADJ_CLAIM_NBR',
       ...
       'CASE QTY', 'DOSE QTY', 'WAC_SHIP', 'ORIGINAL INDICATOR',
       'PROD_DTL_KEY', 'POC PARENT', '_ship_name', '_ship_zip', '_ship_month',
       '_ship_year'],
      dtype='object', length=131)

In [ ]:
# import os
# import pandas as pd

# def load_copay(path: str = None) -> pd.DataFrame:
#     path = path or COPAY_PATH

#     # 0. Basic sanity checks first — catches the OneDrive/locked-file case.
#     if not os.path.exists(path):
#         raise FileNotFoundError(f"File not found: {path}")
#     size = os.path.getsize(path)
#     if size == 0:
#         raise ValueError(f"File is empty (0 bytes) — likely not downloaded/synced: {path}")
#     print(f"Reading {path} ({size:,} bytes)")

#     # 1. Try the fast C engine; fall back to the python engine on tokenizing errors.
#     read_kwargs = dict(low_memory=False)
#     try:
#         df = pd.read_csv(path, **read_kwargs)
#     except pd.errors.ParserError as e:
#         print(f"[warn] C engine failed ({e}); retrying with engine='python'")
#         df = pd.read_csv(
#             path,
#             engine="python",
#             on_bad_lines="warn",   # skip & report malformed rows instead of crashing
#             **read_kwargs,
#         )
#     except UnicodeDecodeError as e:
#         print(f"[warn] encoding error ({e}); retrying with latin-1")
#         df = pd.read_csv(path, encoding="latin-1", engine="python",
#                          on_bad_lines="warn", **read_kwargs)

#     # --- dates ---
#     df["fill_date"] = pd.to_datetime(df["DATE_OF_FILL"], errors="coerce")
#     # ... keep the rest of your original load_copay logic below ...

#     return df

In [27]:
df = load_copay(COPAY_PATH)
df = attach_pharmacy_type(df)
ref_month = df['month'].max()
ris = load_risrx(RISRX_PATH, data_min=df['fill_date'].min(), data_max=df['fill_date'].max())
print(f"{len(df):,} claims | {df['pharmacy_key'].nunique():,} pharmacies | "
      f"{df['fill_date'].min().date()} -> {df['fill_date'].max().date()}")
print(f"snapshot (reference) month: {pd.Timestamp(ref_month).date()}")
print(f"RisRx: {ris['pharmacy_key'].nunique()} pharmacies")

964,206 claims | 31,877 pharmacies | 2018-05-31 -> 2026-07-05
snapshot (reference) month: 2026-07-01
RisRx: 150 pharmacies


In [20]:
# after: df = load_copay("COPAY_EUCRISA_CLEAN.csv")
red = df[(~df["is_reversal"]) & (df["QTY_DISPENSED"] > 0)]
print("Total claims (modeled):", len(red))
print("Raw rows:", len(df))

Total claims (modeled): 964206
Raw rows: 964206


In [21]:
print("Range:", df["fill_date"].min().date(), "→", df["fill_date"].max().date())

Range: 2018-05-31 → 2026-07-05


## 3. Rule engine → labels
Run signals 1–5 & 7 to assemble the base scorecard. `signals_triggered > 0` is the
label the supervised models train against.

In [28]:
score, detail = run_rule_engine(df, ris)
s1, s2, s7 = detail['s1'], detail['s2'], detail['s7']
# s7 comes from: score, detail = run_rule_engine(df, ris); s7 = detail['s7']

# keep only the pharmacies Signal 7 actually flagged
s7_flagged = s7[s7["signal7_popup"]].copy()

# order columns nicely for the sheet
cols = ["pharmacy_key", "signal7_popup",
        "signal7_dormant_reactivation", "reactivation_window_vol", "dormancy_months",
        "signal7_new_high_volume", "new_window_vol",
        "s7_best_gap_months", "s7_best_gap_burst",
        "signal7_event_month", "s7_best_episode_month", "total_claims"]
s7_flagged = s7_flagged[[c for c in cols if c in s7_flagged.columns]]

# sort so the biggest reactivations/launches are on top
s7_flagged = s7_flagged.sort_values(
    ["reactivation_window_vol", "new_window_vol"], ascending=False)

s7_flagged.to_excel("Signal7_Flagged_Pharmacies.xlsx", index=False)
print(f"Wrote Signal7_Flagged_Pharmacies.xlsx : {len(s7_flagged):,} pharmacies")


Signal 1 (CPU vs chain baseline) ...
  flagged: 3709
Signal 2 (cost per claim at max) ...
  flagged: 40
Signal 3 (CPU pre/post RisRx) ...
  flagged rebounds: 1
  flagged pharmacies: 3555
Signal 7 (popup/dormant) ...
  flagged pharmacies: 330
Signal 5 (high HCP utilization at 3,838 flagged pharmacies) ...
  flagged pharmacies: 1245
Building pharmacy features ...

Rule engine: 3,838 pharmacies tripped >=1 signal; 1,288 tripped >=2.
    signal1_cpu_above_chain    3,709
    signal2_cost_at_max        40
    signal3_cpu_rebound        1
    signal5_high_hcp           1,245
    signal7_popup              330
Wrote Signal7_Flagged_Pharmacies.xlsx : 330 pharmacies


In [23]:
# after run_rule_engine
score, detail = run_rule_engine(df, ris)
print("Pharmacies (modeled):", len(score))

Signal 1 (CPU vs chain baseline) ...
1.7715170285200323
  flagged: 3709
Signal 2 (cost per claim at max) ...
  flagged: 40
Signal 3 (CPU pre/post RisRx) ...
  flagged rebounds: 1
  flagged pharmacies: 3555
Signal 7 (popup/dormant) ...
  flagged pharmacies: 330
Signal 5 (high HCP utilization at 3,838 flagged pharmacies) ...
  flagged pharmacies: 1245
Building pharmacy features ...

Rule engine: 3,838 pharmacies tripped >=1 signal; 1,288 tripped >=2.
    signal1_cpu_above_chain    3,709
    signal2_cost_at_max        40
    signal3_cpu_rebound        1
    signal5_high_hcp           1,245
    signal7_popup              330
Pharmacies (modeled): 31877


In [29]:
import nurtec_pipeline as P
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# --- assumes `score` already exists from run_rule_engine ---
# (score, detail = run_rule_engine(df, ris) must have run above)

# We also need proximity/closest_signal for the lookalike breakdown:
hv = detail["s7"].attrs["high_vol_threshold"]

data = score.copy()
data["label"] = (data["signals_triggered"] > 0).astype(int)

X = data[P.CLF_FEATURES].replace([np.inf, -np.inf], 0).fillna(0)
y = data["label"].values
idx = np.arange(len(data))

# ---------- STAGE 1: hold out 10% OOS ----------
X_dev, X_oos, y_dev, y_oos, idx_dev, idx_oos = train_test_split(
    X, y, idx, test_size=0.10, random_state=P.RANDOM_STATE, stratify=y)

# ---------- STAGE 2: split the 90% dev into 70/30 -> 63/27 of total ----------
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dev, y_dev, test_size=0.30, random_state=P.RANDOM_STATE, stratify=y_dev)

print("="*55)
print("THREE-WAY SPLIT")
print("="*55)
n = len(data)
print(f"Total pharmacies : {n:,}")
print(f"Train : {len(y_tr):,} ({len(y_tr)/n:.0%})")
print(f"Test  : {len(y_te):,} ({len(y_te)/n:.0%})")
print(f"OOS   : {len(y_oos):,} ({len(y_oos)/n:.0%})")

# ---------- TRAIN on the 63% train split only ----------
clf = RandomForestClassifier(
    n_estimators=400, max_depth=None, min_samples_leaf=5,
    class_weight="balanced", n_jobs=-1, random_state=P.RANDOM_STATE)
clf.fit(X_tr, y_tr)

# ---------- metrics on TEST (27%) ----------
proba_te = clf.predict_proba(X_te)[:, 1]
print(f"\nTEST (27%)  ROC-AUC {roc_auc_score(y_te, proba_te):.3f} | "
      f"PR-AUC {average_precision_score(y_te, proba_te):.3f}")

# ---------- metrics + counts on OOS (10%) ----------
proba_oos = clf.predict_proba(X_oos)[:, 1]
print(f"OOS  (10%)  ROC-AUC {roc_auc_score(y_oos, proba_oos):.3f} | "
      f"PR-AUC {average_precision_score(y_oos, proba_oos):.3f}")

# attach OOS predictions back to the pharmacy rows
oos = data.iloc[idx_oos].copy()
oos["signal_probability"] = proba_oos
oos_pred1 = oos[oos["signal_probability"] >= 0.50]

tp        = int((oos_pred1["signals_triggered"] > 0).sum())
lookalike = int((oos_pred1["signals_triggered"] == 0).sum())

print("\n" + "="*55)
print("OOS RESULTS (for the slide)")
print("="*55)
print(f"OOS set size          : {len(oos):,}")
print(f"Predicted class 1     : {len(oos_pred1):,}")
print(f"  True positives      : {tp:,}  (triggered >=1 rule)")
print(f"  Lookalikes          : {lookalike:,}  (no rule, high score)")

# RisRx overlap in OOS predicted class 1
on_ris = int(oos_pred1["on_risrx"].sum())
print(f"RisRx overlap (OOS pred 1): {on_ris}/{on_ris} caught")

# ---------- lookalike closest-signal breakdown (OOS only) ----------
oos_prox = P.signal_proximity(oos, detail["s3"], hv)
oos_look = oos_prox[(oos_prox["signal_probability"] >= 0.50) &
                    (oos_prox["signals_triggered"] == 0)]
print("\nOOS lookalikes by closest signal:")
print(oos_look["closest_signal"].value_counts().to_string())

THREE-WAY SPLIT
Total pharmacies : 31,877
Train : 20,082 (63%)
Test  : 8,607 (27%)
OOS   : 3,188 (10%)

TEST (27%)  ROC-AUC 0.966 | PR-AUC 0.765
OOS  (10%)  ROC-AUC 0.969 | PR-AUC 0.797

OOS RESULTS (for the slide)
OOS set size          : 3,188
Predicted class 1     : 633
  True positives      : 359  (triggered >=1 rule)
  Lookalikes          : 274  (no rule, high score)
RisRx overlap (OOS pred 1): 4/4 caught

OOS lookalikes by closest signal:
closest_signal
Signal 7 (Popup)           254
Signal 2 (Cost at max)      18
Signal 1 (CPU vs chain)      2


In [31]:
import nurtec_pipeline as P
import pandas as pd

# recompute OOS proximity cleanly from the OOS predictions
hv = detail["s7"].attrs["high_vol_threshold"]
oos_prox = P.signal_proximity(oos, detail["s3"], hv)   # `oos` from your OOS cell (has signal_probability)

# subsets
oos_pred1 = oos_prox[oos_prox["signal_probability"] >= 0.50].copy()
oos_tp    = oos_pred1[oos_pred1["signals_triggered"] > 0].copy()
oos_look  = oos_pred1[oos_pred1["signals_triggered"] == 0].copy()

# only keep columns that actually exist (guards against KeyError)
wanted = ["pharmacy_key", "total_claims", "signal_probability",
          "signals_triggered", "closest_signal", "closest_proximity",
          "closest_reason", "on_risrx"]
cols = [c for c in wanted if c in oos_prox.columns]

with pd.ExcelWriter("Eucrisa_OOS_Validation.xlsx", engine="openpyxl") as xw:
    oos_pred1[cols].sort_values("signal_probability", ascending=False)\
        .to_excel(xw, "OOS_Predicted_Class1", index=False)
    oos_tp[cols].to_excel(xw, "OOS_True_Positives", index=False)
    oos_look[cols].to_excel(xw, "OOS_Lookalikes", index=False)

print(f"Wrote Eucrisa_OOS_Validation.xlsx  ({len(oos_pred1)} / {len(oos_tp)} / {len(oos_look)})")
print("Columns written:", cols)

Wrote Eucrisa_OOS_Validation.xlsx  (633 / 359 / 274)
Columns written: ['pharmacy_key', 'total_claims', 'signal_probability', 'signals_triggered', 'closest_signal', 'closest_proximity', 'closest_reason', 'on_risrx']


## 4. Fit the composite anomaly model
IsolationForest on the behavioral feature block. We persist the scaler **and** the raw
score min/max so notebook 02 normalises new data onto the identical 0–1 scale.

In [8]:
anomaly_art = fit_anomaly_model(score)
score = apply_anomaly_model(score, anomaly_art)
print(f"anomalies flagged (training pop): {int(score['is_anomaly'].sum()):,}")

anomalies flagged (training pop): 1,594


## 5. Train + evaluate the lookalike classifier
`train_signal_classifier` holds out a stratified test split for honest ROC-AUC / PR-AUC.
For deployment we then **refit on the full population** (`fit_classifier_full`) so the
shipped model has seen every row.

In [9]:
score, clf_metrics, importances, _clf_eval = train_signal_classifier(score)
print(f"population {clf_metrics['n_total']:,} | positives {clf_metrics['n_positive']:,} "
      f"({clf_metrics['positive_rate']:.1%})")
print(f"hold-out ROC-AUC {clf_metrics['roc_auc']:.3f} | PR-AUC {clf_metrics['pr_auc']:.3f}")
print(clf_metrics['report'])

classifier = fit_classifier_full(score) if REFIT_CLASSIFIER_ON_FULL else _clf_eval
importances.head(8)

population 31,877 | positives 3,838 (12.0%)
hold-out ROC-AUC 0.969 | PR-AUC 0.785
              precision    recall  f1-score   support

           0      0.990     0.903     0.945      8412
           1      0.569     0.936     0.708      1152

    accuracy                          0.907      9564
   macro avg      0.780     0.919     0.826      9564
weighted avg      0.940     0.907     0.916      9564



,feature,importance
0,avg_cpu,0.301048
1,avg_cpu_pct_wac,0.216373
2,total_benefit,0.136859
3,total_claims,0.102673
4,std_cpu,0.090340
5,total_units,0.066859
6,claims_per_month,0.035314
7,n_months,0.033481


## 6. Fit per-signal propensity models
One-vs-rest RandomForest per signal with ≥ `PROPENSITY_MIN_POSITIVES` positives. Signals
below the bar (typically S2, S3) are intentionally skipped — their column is NaN at scoring.

In [10]:
propensity_models = fit_per_signal_propensity(score)
fitted = sorted(propensity_models)
skipped = [p for p in ['p_signal1','p_signal2','p_signal3','p_signal4','p_signal5','p_signal7'] if p not in propensity_models]
print('fitted :', fitted)
print('skipped:', skipped, '(too few positives)')

fitted : ['p_signal1', 'p_signal5', 'p_signal7']
skipped: ['p_signal2', 'p_signal3', 'p_signal4'] (too few positives)


## 7. Persist the model bundle
Everything notebook 02 needs in one file (`models/nurtec_models.joblib`) plus a human-readable
`model_metadata.json` for provenance.

In [11]:
bundle = {
    'classifier': classifier,
    'anomaly': anomaly_art,
    'propensity': propensity_models,
    'importances': importances,
    'metadata': {
        'trained_at': datetime.now().isoformat(timespec='seconds'),
        'ref_month': str(pd.Timestamp(ref_month).date()),
        'n_pharmacies': int(clf_metrics['n_total']),
        'n_positive': int(clf_metrics['n_positive']),
        'positive_rate': clf_metrics['positive_rate'],
        'roc_auc': clf_metrics['roc_auc'],
        'pr_auc': clf_metrics['pr_auc'],
        'clf_features': CLF_FEATURES,
        'anomaly_cols': anomaly_art['model_cols'],
        'propensity_signals': fitted,
        'propensity_min_positives': PROPENSITY_MIN_POSITIVES,
        'lookalike_prob_threshold': LOOKALIKE_PROB_THRESHOLD,
        'chain_baseline_cpu': float(s1.attrs.get('chain_baseline_cpu', float('nan'))),
        'band_floor': float(s2.attrs.get('band_floor', float('nan'))),
        'program_max': float(s2.attrs.get('program_max', float('nan'))),
        'high_vol_threshold': float(s7.attrs.get('high_vol_threshold', float('nan'))),
    },
}
save_model_bundle(bundle)

Saved models\eucrisa_models.joblib
Saved models\model_metadata.json
